In [ ]:
%matplotlib widget

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, RadioButtons, HTML, HTMLMath, VBox, HBox, Layout
from IPython.display import display

plt.ioff()

display(HTML("""
<style>
.container{width:98%!important;max-width:none!important}
.output_area,.output_subarea,.jp-Cell-outputWrapper,.jp-OutputArea,.jp-OutputArea-child,
.jp-OutputArea-output,.widget-output,.jupyter-widgets-output-area,.widget-box{
max-width:none!important;height:auto!important;max-height:none!important;overflow:visible!important}
.output_scroll{height:auto!important;max-height:none!important;overflow:visible!important;box-shadow:none!important}
.jupyter-matplotlib,.jupyter-matplotlib-figure{overflow:visible!important;resize:none!important}
.vec-title{font-family:Arial;font-size:20px;font-weight:bold;color:#6f3fa0}
.vec-label{font-family:Arial;font-size:14px;font-weight:bold}
.vec-value{font-family:Arial;font-size:14px;font-weight:bold;color:#0b3d91}
.vec-radio .widget-radio-box{display:flex!important;flex-direction:row!important;gap:22px!important}
.vec-radio>label{display:none!important}
</style>
"""))

documentation = HTML("""
<div style="width:1180px;font-family:Arial;font-size:15px;line-height:1.5;margin-bottom:10px">
<div class="vec-title" style="margin-bottom:8px">Linear Dependence, Span and Basis</div>
<div style="margin-bottom:5px">
A set of vectors is linearly independent when the only linear combination producing
the zero vector is the trivial one.
</div>
<div style="margin-bottom:5px">
In R², two vectors form a basis when the determinant of the matrix having them as
columns is nonzero. Its absolute value is the area of the generated parallelogram.
</div>
<div style="margin-bottom:5px">
In R³, three vectors form a basis when the corresponding 3×3 matrix has rank three,
equivalently when its determinant is nonzero. Its absolute value is the generated volume.
</div>
<div><b>This notebook:</b> computes determinant, rank, linear dependence, span and
basis properties symbolically using SymPy.</div>
</div>
""")

space_selector = RadioButtons(
    options=[('R²','2D'),('R³','3D')],
    value='2D', description='', layout=Layout(width='260px')
)
space_selector.add_class('vec-radio')

def slider(v):
    return IntSlider(min=-5,max=5,step=1,value=v,readout=False,
                     continuous_update=True,layout=Layout(width='190px'))

# ---------------- 2D sliders ----------------
u1,u2,v1,v2 = slider(3),slider(1),slider(1),slider(3)
u1v,u2v,v1v,v2v = [HTML() for _ in range(4)]

# ---------------- 3D sliders ----------------
a1,a2,a3 = slider(3),slider(0),slider(1)
b1,b2,b3 = slider(1),slider(3),slider(0)
c1,c2,c3 = slider(0),slider(1),slider(3)
a1v,a2v,a3v,b1v,b2v,b3v,c1v,c2v,c3v = [HTML() for _ in range(9)]

def row(label,s,w):
    return HBox([
        HTML(f'<div class="vec-label">{label}</div>',layout=Layout(width='45px')),
        s,w
    ],layout=Layout(width='270px',height='34px',align_items='center'))

controls_2d = HBox([
    VBox([HTML('<div class="vec-title" style="font-size:17px">Vector u</div>'),
          row('u₁:',u1,u1v),row('u₂:',u2,u2v)]),
    VBox([HTML('<div class="vec-title" style="font-size:17px">Vector v</div>'),
          row('v₁:',v1,v1v),row('v₂:',v2,v2v)])
],layout=Layout(width='620px',gap='12px'))

controls_3d = HBox([
    VBox([HTML('<div class="vec-title" style="font-size:17px">Vector u</div>'),
          row('u₁:',a1,a1v),row('u₂:',a2,a2v),row('u₃:',a3,a3v)]),
    VBox([HTML('<div class="vec-title" style="font-size:17px">Vector v</div>'),
          row('v₁:',b1,b1v),row('v₂:',b2,b2v),row('v₃:',b3,b3v)]),
    VBox([HTML('<div class="vec-title" style="font-size:17px">Vector w</div>'),
          row('w₁:',c1,c1v),row('w₂:',c2,c2v),row('w₃:',c3,c3v)])
],layout=Layout(width='930px',gap='8px'))

controls_3d.layout.display='none'

parameters_panel = VBox([
    HTML('<div class="vec-title" style="margin-bottom:6px">Vector Parameters</div>'),
    space_selector,controls_2d,controls_3d
],layout=Layout(width='950px',padding='10px 14px',border='1px solid #d2c2df'))

symbolic_math = HTMLMath()
symbolic_text = HTML()

symbolic_panel = VBox([
    HTML('<div class="vec-title" style="margin-bottom:6px">Symbolic Results</div>'),
    symbolic_math,symbolic_text
],layout=Layout(width='950px',padding='10px 14px',border='1px solid #d2c2df'))

# ============================================================
# 2D FIGURE
# ============================================================

fig2,ax2 = plt.subplots(figsize=(7.2,5.2))
fig2.canvas.header_visible=False
fig2.canvas.footer_visible=False
fig2.canvas.toolbar_visible=False
fig2.canvas.layout=Layout(width='720px',height='520px')

ax2.set_title('Geometry of Two Vectors in R²',fontsize=14,fontweight='bold',color='#6f3fa0')
ax2.set_xlabel('x₁'); ax2.set_ylabel('x₂')
ax2.set_xlim(-6,6); ax2.set_ylim(-6,6)
ax2.set_aspect('equal',adjustable='box')
ax2.axhline(0,linewidth=.8); ax2.axvline(0,linewidth=.8)
ax2.grid(True,linestyle=':',alpha=.4)

u2line, = ax2.plot([0,3],[0,1],linewidth=2.4,marker='o',label='u')
v2line, = ax2.plot([0,1],[0,3],linewidth=2.4,marker='o',label='v')
parline, = ax2.plot([],[],linewidth=1.5,linestyle='--')
spanline, = ax2.plot([],[],linewidth=1.5,linestyle=':')
ax2.legend(loc='upper right')
fig2.subplots_adjust(left=.10,right=.97,top=.90,bottom=.12)

# ============================================================
# 3D FIGURE
# ============================================================

fig3 = plt.figure(figsize=(7.2,5.5))
ax3 = fig3.add_subplot(111,projection='3d')
fig3.canvas.header_visible=False
fig3.canvas.footer_visible=False
fig3.canvas.toolbar_visible=False
fig3.canvas.layout=Layout(width='720px',height='550px')
fig3.canvas.layout.display='none'

ax3.set_title('Geometry of Three Vectors in R³',fontsize=14,fontweight='bold',color='#6f3fa0')
ax3.set_xlabel('x₁'); ax3.set_ylabel('x₂'); ax3.set_zlabel('x₃')
ax3.set_xlim(-6,6); ax3.set_ylim(-6,6); ax3.set_zlim(-6,6)

u3line, = ax3.plot([0,3],[0,0],[0,1],linewidth=2.5,marker='o',label='u')
v3line, = ax3.plot([0,1],[0,3],[0,0],linewidth=2.5,marker='o',label='v')
w3line, = ax3.plot([0,0],[0,1],[0,3],linewidth=2.5,marker='o',label='w')
ax3.legend(loc='upper right')

figure_holder = VBox([fig2.canvas,fig3.canvas],layout=Layout(width='740px'))

# ============================================================
# SYMBOLIC COMPUTATION
# ============================================================

def compute2():
    u=sp.Matrix([u1.value,u2.value])
    v=sp.Matrix([v1.value,v2.value])
    A=sp.Matrix.hstack(u,v)
    return u,v,A,sp.simplify(A.det()),A.rank()

def compute3():
    u=sp.Matrix([a1.value,a2.value,a3.value])
    v=sp.Matrix([b1.value,b2.value,b3.value])
    w=sp.Matrix([c1.value,c2.value,c3.value])
    A=sp.Matrix.hstack(u,v,w)
    return u,v,w,A,sp.simplify(A.det()),A.rank()

# ============================================================
# UPDATE 2D
# ============================================================

def update2():
    u,v,A,detA,rankA = compute2()
    ux,uy=float(u[0]),float(u[1])
    vx,vy=float(v[0]),float(v[1])

    u2line.set_data([0,ux],[0,uy])
    v2line.set_data([0,vx],[0,vy])

    if detA!=0:
        parline.set_data([0,ux,ux+vx,vx,0],[0,uy,uy+vy,vy,0])
        spanline.set_data([],[])
        conclusion='The vectors are linearly independent. Their span is R² and they form a basis of R².'
    else:
        parline.set_data([],[])
        direction=None
        if u!=sp.zeros(2,1): direction=np.array(u,dtype=float).reshape(-1)
        elif v!=sp.zeros(2,1): direction=np.array(v,dtype=float).reshape(-1)

        if direction is not None and np.linalg.norm(direction)>0:
            direction=direction/np.linalg.norm(direction)
            s=np.array([-8.,8.])
            spanline.set_data(s*direction[0],s*direction[1])
        else:
            spanline.set_data([],[])

        conclusion = (
            'The vectors are linearly dependent. Their span is a one-dimensional subspace (a line) of R².'
            if rankA==1 else
            'Both vectors are zero. Their span contains only the zero vector.'
        )

    symbolic_math.value = (
        r'\('
        r'\mathbf{u}='+sp.latex(u)+
        r'\;\;\;\mathbf{v}='+sp.latex(v)+
        r'\;\;\;\;\;\;A='+sp.latex(A)+
        r'\)'
        r'<div style="height:18px"></div>'
        r'\('
        r'\det(A)='+sp.latex(detA)+
        r'\;\;\;\operatorname{rank}(A)='+str(rankA)+
        r'\;\;\;\text{Area}=|\det(A)|='+sp.latex(abs(detA))+
        r'\)'
    )

    symbolic_text.value = (
        '<div style="font-family:Arial;font-size:14px;line-height:1.5;margin-top:12px">'
        '<b>Conclusion:</b> '+conclusion+'</div>'
    )

    fig2.canvas.draw_idle()

# ============================================================
# UPDATE 3D
# ============================================================

def update3():
    u,v,w,A,detA,rankA = compute3()

    un=np.array(u,dtype=float).reshape(-1)
    vn=np.array(v,dtype=float).reshape(-1)
    wn=np.array(w,dtype=float).reshape(-1)

    u3line.set_data_3d([0,un[0]],[0,un[1]],[0,un[2]])
    v3line.set_data_3d([0,vn[0]],[0,vn[1]],[0,vn[2]])
    w3line.set_data_3d([0,wn[0]],[0,wn[1]],[0,wn[2]])

    if detA!=0:
        conclusion='The three vectors are linearly independent. Their span is R³ and they form a basis of R³.'
    elif rankA==2:
        conclusion='The vectors are linearly dependent. Their span is a two-dimensional subspace (a plane) of R³.'
    elif rankA==1:
        conclusion='The vectors are linearly dependent. Their span is a one-dimensional subspace (a line) of R³.'
    else:
        conclusion='All vectors are zero. Their span contains only the zero vector.'

    symbolic_math.value = (
        r'\('
        r'\mathbf{u}='+sp.latex(u)+
        r'\;\;\mathbf{v}='+sp.latex(v)+
        r'\;\;\mathbf{w}='+sp.latex(w)+
        r'\;\;\;\;\;A='+sp.latex(A)+
        r'\)'
        r'<div style="height:18px"></div>'
        r'\('
        r'\det(A)='+sp.latex(detA)+
        r'\;\;\;\operatorname{rank}(A)='+str(rankA)+
        r'\;\;\;\text{Volume}=|\det(A)|='+sp.latex(abs(detA))+
        r'\)'
    )

    symbolic_text.value = (
        '<div style="font-family:Arial;font-size:14px;line-height:1.5;margin-top:12px">'
        '<b>Conclusion:</b> '+conclusion+'</div>'
    )

    fig3.canvas.draw_idle()

# ============================================================
# SLIDER LABELS + GENERAL UPDATE
# ============================================================

def update_labels():
    pairs=[
        (u1v,u1),(u2v,u2),(v1v,v1),(v2v,v2),
        (a1v,a1),(a2v,a2),(a3v,a3),
        (b1v,b1),(b2v,b2),(b3v,b3),
        (c1v,c1),(c2v,c2),(c3v,c3)
    ]
    for widget,s in pairs:
        widget.value=f'<div class="vec-value">{s.value}</div>'

def update(change=None):
    update_labels()

    if space_selector.value=='2D':
        controls_2d.layout.display='flex'
        controls_3d.layout.display='none'
        fig2.canvas.layout.display='block'
        fig3.canvas.layout.display='none'
        update2()
    else:
        controls_2d.layout.display='none'
        controls_3d.layout.display='flex'
        fig2.canvas.layout.display='none'
        fig3.canvas.layout.display='block'
        update3()

space_selector.observe(update,names='value')

for s in [u1,u2,v1,v2,a1,a2,a3,b1,b2,b3,c1,c2,c3]:
    s.observe(update,names='value')

update()

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
width:950px;padding:10px 13px;border:1px solid #d7c7e5;
font-family:Arial;font-size:14px;line-height:1.55;box-sizing:border-box">

<div style="color:#6f3fa0;font-size:17px;font-weight:bold;margin-bottom:6px">
Interpretation
</div>

<div style="margin-bottom:6px">
In R², two independent vectors generate a nonzero parallelogram area and span the
whole plane. When the determinant becomes zero, the area collapses and the vectors
belong to the same line.
</div>

<div style="margin-bottom:6px">
In R³, three independent vectors generate a nonzero volume and span the whole
three-dimensional space. A zero determinant means that the vectors belong to a
lower-dimensional subspace.
</div>

<div>
The determinant and rank are calculated directly by SymPy from the vector components.
</div>
</div>
""")

display(
    VBox(
        [documentation,parameters_panel,symbolic_panel,figure_holder,interpretation],
        layout=Layout(width='1180px',gap='10px',align_items='flex-start')
    )
)